<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보조 코드 by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 부록 E: LoRA를 이용한 파라미터 효율적 미세조정

In [ ]:
from importlib.metadata import version

pkgs = ["matplotlib",
        "numpy",
        "tiktoken",
        "torch",
        "tensorflow", # OpenAI의 사전훈련된 가중치용
        "pandas"      # 데이터셋 로딩
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

## E.1 LoRA 소개

- 이 섹션에는 코드가 없습니다.
- 저순위 적응(Low-rank adaptation, LoRA)은 모델 매개변수의 작고 저순위인 부분집합만 조정하여 사전훈련된 모델을 특정한, 종종 더 작은 데이터셋에 더 잘 맞도록 수정하는 머신러닝 기법입니다.
- 이 접근법은 작업별 데이터에 대한 대형 모델의 효율적인 미세조정을 가능하게 하여 미세조정에 필요한 계산 비용과 시간을 크게 줄이기 때문에 중요합니다.

- 주어진 층에 대해 큰 가중치 행렬 $W$가 있다고 가정해봅시다.
- 역전파 중에, 훈련 중 손실 함수를 최소화하기 위해 원래 가중치를 얼마나 업데이트하고 싶은지에 대한 정보를 담고 있는 $\Delta W$ 행렬을 학습합니다.
- 일반적인 훈련과 미세조정에서, 가중치 업데이트는 다음과 같이 정의됩니다:

$$W_{\text{updated}} = W + \Delta W$$

- [Hu et al.](https://arxiv.org/abs/2106.09685)이 제안한 LoRA 방법은 가중치 업데이트 $\Delta W$의 근사치인 $\Delta W \approx AB$를 학습하여 이를 계산하는 더 효율적인 대안을 제공합니다.
- 다시 말해, LoRA에서는 $A$와 $B$가 두 개의 작은 가중치 행렬인 다음을 갖습니다:

$$W_{\text{updated}} = W + AB$$

- 아래 그림은 전체 미세조정과 LoRA에 대한 이러한 공식을 나란히 보여줍니다.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-e_compressed/lora-1.webp" width="500px">

- 주의 깊게 보셨다면, 위 그림의 전체 미세조정과 LoRA 표현이 앞서 보여드린 공식과 약간 다르게 보입니다.
- 이는 행렬 곱셈의 분배법칙 때문입니다: 가중치를 업데이트된 가중치와 더할 필요가 없고 별도로 유지할 수 있습니다.
- 예를 들어, $x$가 입력 데이터라면, 일반 미세조정에 대해 다음과 같이 쓸 수 있습니다:

$$x (W+\Delta W) = x W + x \Delta W$$

- 유사하게, LoRA에 대해 다음과 같이 쓸 수 있습니다:

$$x (W+A B) = x W + x A B$$

- LoRA 가중치 행렬을 별도로 유지할 수 있다는 사실이 LoRA를 특히 매력적으로 만듭니다.
- 실제로 이는 LoRA 행렬을 즉시 적용할 수 있기 때문에 사전훈련된 모델의 가중치를 전혀 수정할 필요가 없다는 것을 의미합니다.
- 데이터셋을 설정하고 모델을 로드한 후, 이러한 개념을 덜 추상적으로 만들기 위해 코드에서 LoRA를 구현하겠습니다.

## E.2 데이터셋 준비

- 이 섹션은 데이터셋을 로드하고 준비하기 위해 6장의 코드를 반복합니다.
- 이 코드를 반복하는 대신, 6장 노트북을 열어서 실행한 다음 섹션 E.4의 LoRA 코드를 그곳에 삽입할 수도 있습니다.
- (LoRA 코드는 원래 6장의 마지막 섹션이었지만 6장의 길이 때문에 부록으로 이동되었습니다)
- 유사하게, 지시 미세조정을 위해 7장의 모델에도 LoRA를 적용할 수 있습니다.

In [ ]:
import urllib
from pathlib import Path
import pandas as pd
from previous_chapters import (
    download_and_unzip_spam_data,
    create_balanced_dataset,
    random_split
)
# 만약 `previous_chapters.py` 파일이 로컬에 없다면,
# `llms-from-scratch` PyPI 패키지에서 가져올 수 있습니다.
# 자세한 내용은: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# 예시:
# from llms_from_scratch.ch06 import (
#     download_and_unzip_spam_data,
#     create_balanced_dataset,
#     random_split
# )



url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

try:
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)
except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError) as e:
    print(f"Primary URL failed: {e}. Trying backup URL...")
    url = "https://f001.backblazeb2.com/file/LLMs-from-scratch/sms%2Bspam%2Bcollection.zip"
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
balanced_df = create_balanced_dataset(df)
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)
train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

In [ ]:
import torch
import tiktoken
from previous_chapters import SpamDataset


tokenizer = tiktoken.get_encoding("gpt2")
train_dataset = SpamDataset("train.csv", max_length=None, tokenizer=tokenizer)
val_dataset = SpamDataset("validation.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)
test_dataset = SpamDataset("test.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)

In [ ]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

- 검증 단계로, 데이터 로더를 반복하고 배치가 각각 8개의 훈련 예제를 포함하는지 확인하며, 각 훈련 예제는 120개의 토큰으로 구성되어 있습니다.

In [ ]:
print("Train loader:")
for input_batch, target_batch in train_loader:
    pass

print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions", target_batch.shape)

- 마지막으로, 각 데이터셋의 총 배치 수를 출력해봅시다.

In [ ]:
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

## E.3 모델 초기화

- 이 섹션은 모델을 로드하고 준비하기 위해 6장의 코드를 반복합니다.

In [ ]:
from gpt_download import download_and_load_gpt2
from previous_chapters import GPTModel, load_weights_into_gpt
# 대안:
# from llms_from_scratch.ch04 import GPTModel
# from llms_from_scratch.ch05 import load_weights_into_gpt



CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"

BASE_CONFIG = {
    "vocab_size": 50257,     # 어휘 크기
    "context_length": 1024,  # 컨텍스트 길이
    "drop_rate": 0.0,        # 드롭아웃 비율
    "qkv_bias": True         # Query-key-value 편향
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(model_size=model_size, models_dir="gpt2")

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval();

- 모델이 올바르게 로드되었는지 확인하기 위해, 일관된 텍스트를 생성하는지 다시 한 번 확인해봅시다.

In [ ]:
from previous_chapters import (
    generate_text_simple,
    text_to_token_ids,
    token_ids_to_text
)


text_1 = "Every effort moves you"

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_1, tokenizer),
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

- 그런 다음, 출력층을 교체하여 6장과 유사하게 분류 미세조정을 위한 모델을 준비합니다.

In [ ]:
torch.manual_seed(123)

num_classes = 2
model.out_head = torch.nn.Linear(in_features=768, out_features=num_classes)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 참고:
# 다음 줄들의 주석을 해제하면 해당되는 경우 Apple Silicon 칩에서 코드가 실행되며,
# 이는 Apple CPU보다 약 1.2배 빠릅니다 (M3 MacBook Air에서 측정).
# 하지만 결과 손실 값이 약간 다를 수 있습니다.

#if torch.cuda.is_available():
#    device = torch.device("cuda")
#elif torch.backends.mps.is_available():
#    device = torch.device("mps")
#else:
#    device = torch.device("cpu")
#
# print(f"Using {device} device.")

model.to(device);  # nn.Module 클래스의 경우 model = model.to(device) 할당이 필요하지 않음

- 마지막으로, 미세조정되지 않은 모델의 초기 분류 정확도를 계산해봅시다 (이것이 약 50% 정도일 것으로 예상되며, 이는 모델이 아직 스팸과 스팸이 아닌 메시지를 안정적으로 구별할 수 없다는 의미입니다).

In [ ]:
from previous_chapters import calc_accuracy_loader
# 대안:
# from llms_from_scratch.ch06 import calc_accuracy_loader



torch.manual_seed(123)
train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

## E.4 LoRA를 이용한 파라미터 효율적 미세조정

- $\alpha$ 스케일링 하이퍼파라미터와 순위($r$) 하이퍼파라미터와 함께 행렬 $A$와 $B$를 생성하는 LoRALayer를 초기화하는 것부터 시작합니다.
- 이 층은 입력을 받아들이고 아래 그림에서 보여주는 것처럼 해당하는 출력을 계산할 수 있습니다.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-e_compressed/lora-2.webp" width="200px">

위 그림에 묘사된 이 LoRA 층을 코드로 나타내면 다음과 같습니다.

In [ ]:
import math

class LoRALayer(torch.nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = torch.nn.Parameter(torch.empty(in_dim, rank))
        torch.nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))  # 표준 가중치 초기화와 유사
        self.B = torch.nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        x = self.alpha * (x @ self.A @ self.B)
        return x

- 위 코드에서 `rank`는 행렬 $A$와 $B$의 내부 차원을 제어하는 하이퍼파라미터입니다.
- 다시 말해, 이 매개변수는 LoRA에 의해 도입되는 추가 매개변수의 수를 제어하며, 모델 적응성과 매개변수 효율성 간의 균형을 결정하는 핵심 요소입니다.
- 두 번째 하이퍼파라미터인 `alpha`는 저순위 적응의 출력에 적용되는 스케일링 하이퍼파라미터입니다.
- 이는 본질적으로 적응된 층의 출력이 적응되는 층의 원래 출력에 영향을 미칠 수 있는 정도를 제어합니다.
- 이는 저순위 적응이 층의 출력에 미치는 영향을 조절하는 방법으로 볼 수 있습니다.
- 지금까지 위에서 구현한 `LoRALayer` 클래스는 층 입력 $x$를 변환할 수 있게 해줍니다.
- 그러나 LoRA에서는 일반적으로 아래 그림에서 보여주는 것처럼 가중치 업데이트가 기존 사전훈련된 가중치에 적용되도록 기존 `Linear` 층을 교체하는 데 관심이 있습니다.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-e_compressed/lora-3.webp" width="200px">

- 위 그림에서 보여준 원래 `Linear` 층 가중치를 통합하기 위해, 이전에 구현한 LoRALayer를 사용하고 신경망에서 기존 `Linear` 층을 교체하는 데 사용할 수 있는 `LinearWithLoRA` 층을 아래에 구현합니다. 예를 들어, LLM의 자기 어텐션 모듈이나 피드포워드 모듈에서 사용할 수 있습니다.

In [ ]:
class LinearWithLoRA(torch.nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)

- LoRA 층에서 가중치 행렬 $B$(`LoRALayer`의 `self.B`)를 0 값으로 초기화하기 때문에, $A$와 $B$ 사이의 행렬 곱셈은 0으로 구성된 행렬이 되고 원래 가중치에 영향을 주지 않습니다 (원래 가중치에 0을 더해도 수정되지 않기 때문입니다).

- 앞서 정의한 GPT 모델에서 LoRA를 시도해보기 위해, 모델의 모든 `Linear` 층을 새로운 `LinearWithLoRA` 층으로 교체하는 `replace_linear_with_lora` 함수를 정의합니다.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-e_compressed/lora-4.webp" width="400px">

In [ ]:
def replace_linear_with_lora(model, rank, alpha):
    for name, module in model.named_children():
        if isinstance(module, torch.nn.Linear):
            # Linear 층을 LinearWithLoRA로 교체
            setattr(model, name, LinearWithLoRA(module, rank, alpha))
        else:
            # 자식 모듈에 동일한 함수를 재귀적으로 적용
            replace_linear_with_lora(module, rank, alpha)

- 그런 다음 원래 모델 매개변수를 고정하고 `replace_linear_with_lora`를 사용하여 해당 `Linear` 층을 아래 코드로 교체합니다.
- 이렇게 하면 LLM의 `Linear` 층이 `LinearWithLoRA` 층으로 교체됩니다.

In [ ]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters before: {total_params:,}")

for param in model.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters after: {total_params:,}")

In [ ]:
replace_linear_with_lora(model, rank=16, alpha=16)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable LoRA parameters: {total_params:,}")

- 보시다시피, LoRA를 사용할 때 훈련 가능한 매개변수 수를 거의 50배 줄였습니다.
- 이제 층이 의도한 대로 수정되었는지 모델 아키텍처를 출력하여 다시 한 번 확인해봅시다.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(model)

- 위의 모델 아키텍처를 기반으로, 모델이 이제 새로운 `LinearWithLoRA` 층을 포함하고 있음을 알 수 있습니다.
- 또한, 행렬 $B$를 0으로 초기화했기 때문에, 초기 모델 성능은 이전과 비교하여 변하지 않을 것으로 예상됩니다.

In [ ]:
torch.manual_seed(123)
train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

- 이제 흥미로운 부분으로 넘어가서 6장의 훈련 함수를 재사용하여 모델을 미세조정해봅시다.
- 훈련은 M3 MacBook Air 노트북 컴퓨터에서 약 15분, V100 또는 A100 GPU에서 30초 미만이 걸립니다.

In [ ]:
import time
from previous_chapters import train_classifier_simple
# 대안:
# from llms_from_scratch.ch06 import train_classifier_simple


start_time = time.time()

torch.manual_seed(123)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)

num_epochs = 5
train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

- 마지막으로, 모델을 평가해봅시다.

In [ ]:
from previous_chapters import plot_values
# 대안:
# from llms_from_scratch.ch06 import plot_values

epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_losses))

plot_values(epochs_tensor, examples_seen_tensor, train_losses, val_losses, label="loss")

- 이전에 `eval_iter=5` 설정을 통해 5개 배치에서만 정확도 값을 계산했음을 참고하세요. 아래에서는 전체 데이터셋에 대한 정확도를 계산합니다.

In [ ]:
train_accuracy = calc_accuracy_loader(train_loader, model, device)
val_accuracy = calc_accuracy_loader(val_loader, model, device)
test_accuracy = calc_accuracy_loader(test_loader, model, device)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

- 위의 비교적 높은 정확도 값을 기반으로 볼 때, LoRA 미세조정이 성공적이었음을 알 수 있습니다.